## Cell 1 — Notebook Goal and Flow

This notebook demonstrates a full mini-ingestion pipeline from **Random User API** to **MySQL**.

### What this notebook does
1. Reads environment-based configuration
2. Connects to MySQL through SQLAlchemy
3. Calls API with retries (resilient API access)
4. Normalizes nested JSON into a table-friendly DataFrame
5. Applies basic data quality checks
6. Creates a target table if needed
7. Performs idempotent upsert using `user_uuid`
8. Verifies the load with SQL queries

> Run cells in order from top to bottom because each later cell depends on objects created earlier.

In [23]:
# Standard library
import os
from typing import Any, Dict, Optional

# Third-party libraries
import requests
import pandas as pd
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from sqlalchemy import create_engine, text

# =========================
# API Configuration
# =========================
# Reads API base URL from environment if set; otherwise uses Random User default.
API_BASE_URL = os.getenv("API_BASE_URL", "https://randomuser.me")

# Number of users to request per API call (default = 100).
RESULTS_PER_CALL = int(os.getenv("RESULTS_PER_CALL", "100"))

# =========================
# MySQL Configuration
# =========================
# These values are typically injected by Docker Compose env vars.
MYSQL_HOST = os.getenv("MYSQL_HOST", "db")
MYSQL_PORT = int(os.getenv("MYSQL_PORT", "3306"))
MYSQL_DATABASE = os.getenv("MYSQL_DATABASE", "demo")
MYSQL_USER = os.getenv("MYSQL_USER", "demo_user")
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD", "demo_pass")

# Quick visibility so we know exactly which endpoints/DB target are active.
print("API_BASE_URL:", API_BASE_URL)
print("RESULTS_PER_CALL:", RESULTS_PER_CALL)
print("MYSQL:", f"{MYSQL_USER}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}")

API_BASE_URL: https://randomuser.me
RESULTS_PER_CALL: 100
MYSQL: demo_user@db:3306/demo


## Cell 2 — Create MySQL Engine

This cell builds the SQLAlchemy connection URL and validates that MySQL is reachable.

In [25]:
# SQLAlchemy URL format for MySQL + PyMySQL driver.
mysql_url = (
    f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}"
    f"@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}"
)

# pool_pre_ping=True helps avoid stale connection issues in long-running sessions.
engine = create_engine(mysql_url, pool_pre_ping=True)

# Connectivity sanity test.
with engine.connect() as conn:
    conn.execute(text("SELECT 1"))

print("✅ Connected to MySQL")

✅ Connected to MySQL


## Cell 3 — API Helper Functions with Retry

This section defines reusable API utilities:
- request headers builder
- custom API error type
- `get_json()` with retry/backoff and response validation

In [26]:
import os
import requests
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

# Custom exception used to trigger retries for controlled API failures.
class ApiError(RuntimeError):
    pass

# API requests accept JSON responses.
def build_headers():
    return {"Accept": "application/json"}

# Retry rules:
# - max 5 attempts
# - exponential backoff between attempts
# - retry only on network errors or ApiError
@retry(
    reraise=True,
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=1, min=1, max=10),
    retry=retry_if_exception_type((requests.RequestException, ApiError)),
)
def get_json(path: str, params=None):
    # Normalize slash handling so we do not accidentally build malformed URLs.
    base = (os.getenv("API_BASE_URL") or "").rstrip("/")
    endpoint = (path or "").lstrip("/")
    url = f"{base}/{endpoint}" if endpoint else f"{base}/"

    # Temporary visibility for debugging endpoint construction.
    print("Calling URL:", url)

    # Make HTTP GET call.
    resp = requests.get(url, headers=build_headers(), params=params, timeout=30)

    # Explicitly fail 4xx/5xx with details for easier troubleshooting.
    if resp.status_code >= 400:
        raise ApiError(f"API error {resp.status_code}. URL={resp.url}. Body={resp.text[:300]}")

    # Validate response type before parsing JSON.
    ct = (resp.headers.get("Content-Type") or "").lower()
    if "json" not in ct:
        raise ApiError(f"Non-JSON response. URL={resp.url}. Content-Type={ct}. Body={resp.text[:300]}")

    return resp.json()

In [27]:
# Optional diagnostic cell: confirms API endpoint behavior independently from helper function.
import os, requests

print("API_BASE_URL =", os.getenv("API_BASE_URL"))

url = "https://randomuser.me/api/?results=1"
r = requests.get(url, timeout=30)

print("Status:", r.status_code)
print("Final URL:", r.url)  # useful to detect redirects
print("Content-Type:", r.headers.get("Content-Type"))
print("First 300 chars:\n", r.text[:300])

API_BASE_URL = https://randomuser.me
Status: 200
Final URL: https://randomuser.me/api/?results=1
Content-Type: application/json; charset=utf-8
First 300 chars:
 {"results":[{"gender":"male","name":{"title":"Mr","first":"Emilio","last":"Pastor"},"location":{"street":{"number":7801,"name":"Calle de Ángel García"},"city":"Vigo","state":"Melilla","country":"Spain","postcode":18940,"coordinates":{"latitude":"-30.0004","longitude":"-137.0950"},"timezone":{"offset


## Cell 4 — Extract Data from API

This cell requests a batch of records from Random User and stores the raw JSON payload.

In [28]:
# Random User API supports batching with the `results` query parameter.
# Using '/api/' assumes API_BASE_URL does not already include '/api'.
# If your env already has '/api', switch path to '' to avoid '/api/api'.
raw_response = get_json("/api/", params={"results": RESULTS_PER_CALL, "format": "json"})

# Show top-level keys (e.g., results, info).
raw_response.keys()

Calling URL: https://randomuser.me/api/


dict_keys(['results', 'info'])

## Cell 5 — Normalize JSON to DataFrame

Converts nested JSON user objects into a flat tabular structure for transformation and loading.

In [29]:
# Pull array of user records from payload.
users = raw_response.get("results", [])

# Flatten nested JSON fields into dot-notated columns (e.g., name.first, login.uuid).
df = pd.json_normalize(users)

# Preview normalized data.
df.head()

,gender,email,phone,cell,nat,name.title,name.first,name.last,location.street.number,location.street.name,...,login.sha256,dob.date,dob.age,registered.date,registered.age,id.name,id.value,picture.large,picture.medium,picture.thumbnail
0,female,maelys.clement@example.com,05-95-85-10-43,06-78-58-63-72,FR,Miss,Maëlys,Clement,4441,Place Paul-Duquaire,...,170656d7763e811139aea5eab51016a2d726e25501adee...,1963-07-24T05:36:18.468Z,62,2013-05-19T09:17:52.402Z,12,INSEE,2630667559120 35,https://randomuser.me/api/portraits/women/74.jpg,https://randomuser.me/api/portraits/med/women/...,https://randomuser.me/api/portraits/thumb/wome...
1,female,latife.sinanoglu@example.com,(900)-860-1779,(383)-077-2059,TR,Mrs,Latife,Sinanoğlu,261,Vatan Cd,...,2b42c2fb5297c21c99fd36839d802489e896f65dce4033...,1985-08-23T12:39:35.903Z,40,2003-01-27T11:20:15.555Z,23,,None,https://randomuser.me/api/portraits/women/50.jpg,https://randomuser.me/api/portraits/med/women/...,https://randomuser.me/api/portraits/thumb/wome...
2,male,erlend.rekstad@example.com,54776045,41186451,NO,Mr,Erlend,Rekstad,2248,Tyribakken,...,7cebf456c31d48f07665bef1fa6644ea9fa394c482992c...,1979-08-13T20:22:45.042Z,46,2010-04-09T13:49:42.354Z,15,FN,13087936761,https://randomuser.me/api/portraits/men/46.jpg,https://randomuser.me/api/portraits/med/men/46...,https://randomuser.me/api/portraits/thumb/men/...
3,female,sophie.lavigne@example.com,X50 A38-5358,I23 E74-3272,CA,Miss,Sophie,Lavigne,8286,Queen St,...,a0d6f69fa1169310502cbb719556ca9bf2cafdb5e8d497...,1966-06-14T01:06:33.215Z,59,2016-10-10T23:34:39.692Z,9,SIN,227695814,https://randomuser.me/api/portraits/women/43.jpg,https://randomuser.me/api/portraits/med/women/...,https://randomuser.me/api/portraits/thumb/wome...
4,female,josefina.villanueva@example.com,(637) 433 9756,(682) 768 7999,MX,Mrs,Josefina,Villanueva,9634,Continuación Puebla,...,ec0c890c5318c8947e6951d02c8ec9ff0744ec5c3ba213...,1987-12-16T16:51:42.735Z,38,2016-12-14T01:07:44.104Z,9,NSS,07 25 74 6800 8,https://randomuser.me/api/portraits/women/18.jpg,https://randomuser.me/api/portraits/med/women/...,https://randomuser.me/api/portraits/thumb/wome...


## Cell 6 — Data Quality and Model Shaping

This cell performs data checks and transforms raw columns into a cleaner model for MySQL.

Checks included:
- dataset is not empty
- natural key `login.uuid` exists
- `user_uuid` has no nulls and no duplicates in this batch

In [ ]:
import hashlib
import json
import numpy as np

# Basic ingestion guardrails.
assert not df.empty, "API returned no data"
assert "login.uuid" in df.columns, "Expected natural key field 'login.uuid'"

# Select a subset of source columns to build relational target model.
df_model = df[[
    "login.uuid",
    "gender",
    "name.first",
    "name.last",
    "email",
    "dob.date",
    "dob.age",
    "location.country",
    "phone"
]].copy()

# Rename fields to target schema naming.
df_model.columns = [
    "user_uuid",
    "gender",
    "first_name",
    "last_name",
    "email",
    "dob",
    "age",
    "country",
    "phone"
]

# Convert API DOB string to date-only (compatible with MySQL DATE).
df_model["dob"] = pd.to_datetime(df_model["dob"], utc=True, errors="coerce").dt.date

# Batch-level key quality checks.
assert df_model["user_uuid"].notna().all(), "Missing user_uuid values"
assert df_model["user_uuid"].is_unique, "Duplicate user_uuid values in this batch"

# Build deterministic record hash for deduplication.
hash_columns = [
    "user_uuid", "gender", "first_name", "last_name",
    "email", "dob", "age", "country", "phone"
]

def _normalize_for_hash(value):
    if pd.isna(value):
        return None
    if hasattr(value, "isoformat"):
        return value.isoformat()
    return value

def _compute_record_hash(row):
    payload = {col: _normalize_for_hash(row[col]) for col in hash_columns}
    canonical_json = json.dumps(payload, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    return hashlib.sha256(canonical_json.encode("utf-8")).hexdigest()

df_model["record_hash"] = df_model.apply(_compute_record_hash, axis=1)

# Dedup step: keep first occurrence of each exact record payload.
rows_before_dedup = len(df_model)
df_model = df_model.drop_duplicates(subset=["record_hash"]).reset_index(drop=True)
rows_after_dedup = len(df_model)
duplicates_removed = rows_before_dedup - rows_after_dedup

assert df_model["record_hash"].notna().all(), "Missing record_hash values after hash generation"
assert df_model["record_hash"].is_unique, "record_hash must be unique after dedup"

print(f"Rows before dedup: {rows_before_dedup}")
print(f"Rows after dedup: {rows_after_dedup}")
print(f"Duplicates removed: {duplicates_removed}")

df_model.head()

,user_uuid,gender,first_name,last_name,email,dob,age,country,phone
0,c21c74be-ce79-404c-b9cd-4a58cff9d821,female,Maëlys,Clement,maelys.clement@example.com,1963-07-24,62,France,05-95-85-10-43
1,bc9ea6b4-ab89-4e23-9dd8-ba50f85ae519,female,Latife,Sinanoğlu,latife.sinanoglu@example.com,1985-08-23,40,Turkey,(900)-860-1779
2,0ea048b0-8f01-4d90-acbe-5e50cb319ec7,male,Erlend,Rekstad,erlend.rekstad@example.com,1979-08-13,46,Norway,54776045
3,13e0d0a6-ce96-479c-92f9-6776bba07812,female,Sophie,Lavigne,sophie.lavigne@example.com,1966-06-14,59,Canada,X50 A38-5358
4,b7834c29-f81b-4dc3-9dc3-3dcf6457706b,female,Josefina,Villanueva,josefina.villanueva@example.com,1987-12-16,38,Mexico,(637) 433 9756


## Cell 7 — Ensure Target Table Exists

Creates `random_users` table if missing so the load is repeatable across fresh environments.

In [ ]:
# Idempotent DDL: safe to run multiple times.
create_sql = """
CREATE TABLE IF NOT EXISTS random_users (
  user_uuid VARCHAR(36) PRIMARY KEY,
  record_hash CHAR(64) NOT NULL,
  gender VARCHAR(10),
  first_name VARCHAR(100),
  last_name VARCHAR(100),
  email VARCHAR(255),
  dob DATE,
  age INT,
  country VARCHAR(100),
  phone VARCHAR(50),
  ingested_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  KEY idx_random_users_record_hash (record_hash)
);
"""

# Execute DDL in a transaction block.
with engine.begin() as conn:
    conn.execute(text(create_sql))

print("✅ Ensured table exists: random_users")

## Cell 8 — Upsert into MySQL

This cell loads data with MySQL upsert semantics (`ON DUPLICATE KEY UPDATE`) so re-runs do not create duplicate primary keys.

In [ ]:
from sqlalchemy import Table, MetaData
from sqlalchemy.dialects.mysql import insert as mysql_insert
import numpy as np

# Work on a copy to avoid unexpected side effects on prior dataframe state.
df_model = df_model.copy()

# Quick type check before loading.
print(df_model[["dob", "record_hash"]].head())

# Reflect existing table metadata from MySQL.
metadata = MetaData()
random_users = Table("random_users", metadata, autoload_with=engine)

# Convert DataFrame rows into list-of-dicts for bulk insert.
records = df_model.to_dict(orient="records")

# Build INSERT statement for all records.
stmt = mysql_insert(random_users).values(records)

# Define columns to update when primary key conflict occurs.
upsert_stmt = stmt.on_duplicate_key_update(
    record_hash=stmt.inserted.record_hash,
    gender=stmt.inserted.gender,
    first_name=stmt.inserted.first_name,
    last_name=stmt.inserted.last_name,
    email=stmt.inserted.email,
    dob=stmt.inserted.dob,
    age=stmt.inserted.age,
    country=stmt.inserted.country,
    phone=stmt.inserted.phone
)

# Execute upsert in a transaction.
with engine.begin() as conn:
    conn.execute(upsert_stmt)

print(f"✅ Upserted {len(records)} records into random_users")

## Cell 9 — Verify Data Load

Runs post-load checks: total row count and latest sample rows.

In [ ]:
# Query total rows and a recent sample to validate successful ingestion.
with engine.connect() as conn:
    rows = conn.execute(text("SELECT COUNT(*) AS c FROM random_users")).scalar_one()
    sample = conn.execute(text("SELECT * FROM random_users ORDER BY ingested_at DESC LIMIT 5")).mappings().all()

rows, sample

In [ ]:
# Optional scratch cell for ad-hoc testing.